# 🌿 F-YOLO Hybrid PestVision — CNN + YOLOv8 + Fuzzy Logic

**Target Accuracy: 92–98%**

This notebook trains **three models** and fuses them:
1. **CNN (MobileNetV2)** — global classification (what pest?)
2. **YOLOv8n** — spatial detection (where is it?)
3. **Fuzzy Logic Engine** — severity fusion (how bad?)

---
> ⚠️ **BEFORE RUNNING:** Go to `Runtime → Change Runtime Type → T4 GPU`
> Run cells **ONE BY ONE** — don't use 'Run All'

## 📦 PHASE 0: Environment Setup & GPU Verification

In [ ]:
# ── GPU Verification ──────────────────────────────────────────────────────────
import torch

print('=' * 60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU DETECTED: {gpu_name}')
    print(f'   VRAM: {gpu_mem:.1f} GB')
    DEVICE = 0
else:
    print('❌ NO GPU FOUND!')
    print('   Go to: Runtime → Change Runtime Type → GPU (T4)')
    print('   Then "Disconnect and delete runtime" and try again.')
    DEVICE = 'cpu'  # will be slow but won't crash
print('=' * 60)

In [ ]:
# ── Install All Dependencies ──────────────────────────────────────────────────
print('Installing packages — this takes ~2 minutes...')
!pip install -q ultralytics gdown pyyaml deep-translator scikit-fuzzy
!pip install -q tensorflow==2.15.0  # stable TF for MobileNetV2
print('\n✅ All packages installed.')

## 📂 PHASE 1: Data Acquisition & Validation

In [ ]:
# ── Data Reset & Download ─────────────────────────────────────────────────────
import os, shutil, zipfile

# Purge any corrupted previous environments
!rm -rf /content/dataset /content/runs
!mkdir -p /content/dataset

print('Downloading dataset from Google Drive...')
result = os.system('gdown "1FIe0mLC8xFkuDuU2L4SzdT_b3-7OkhYF" -O /content/dataset.zip')

# ── Validate the zip before extracting ───────────────────────────────────────
zip_ok = False
if os.path.exists('/content/dataset.zip'):
    zip_size = os.path.getsize('/content/dataset.zip')
    print(f'Downloaded: {zip_size / 1e6:.1f} MB')
    try:
        with zipfile.ZipFile('/content/dataset.zip', 'r') as z:
            test = z.testzip()
            if test is None:
                zip_ok = True
                print('✅ ZIP integrity: VALID')
            else:
                print(f'❌ ZIP corrupted at: {test}')
    except Exception as e:
        print(f'❌ ZIP error: {e}')
else:
    print('❌ Download failed — check that the Google Drive link is set to "Anyone with link"')

if zip_ok:
    !unzip -q -o /content/dataset.zip -d /content/dataset/
    !rm -rf /content/dataset/__MACOSX
    # Fix trailing-space folder name
    bad_dir  = '/content/dataset/sc_project '
    good_dir = '/content/dataset/sc_project'
    if os.path.exists(bad_dir):
        shutil.move(bad_dir, good_dir)
        print('Fixed trailing-space directory name.')
    print('✅ Dataset ready at /content/dataset/sc_project')
else:
    print('\n⚠️  MANUAL UPLOAD FALLBACK:')
    print('   1. Upload your dataset zip manually using the Files panel (left sidebar)')
    print('   2. Rename the variable below and re-run')

In [ ]:
# ── YAML Translation: Chinese → English ───────────────────────────────────────
import yaml

yaml_path = '/content/dataset/sc_project/data.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    data = yaml.safe_load(f)

print(f'Found {len(data["names"])} original classes.')

# Only translate if names look like Chinese characters
needs_translation = any(any('\u4e00' <= c <= '\u9fff' for c in name) for name in data['names'][:5])

if needs_translation:
    from deep_translator import GoogleTranslator
    print('Translating Chinese class names to English...')
    translator = GoogleTranslator(source='zh-CN', target='en')
    translated = []
    for i, name in enumerate(data['names']):
        try:
            translated.append(translator.translate(name))
        except Exception:
            translated.append(name)  # keep original if translation fails
        if (i+1) % 20 == 0:
            print(f'  Translated {i+1}/{len(data["names"])}...')
    data['names'] = translated
    print('Translation complete.')
else:
    print('Names already in English — skipping translation.')

data['train'] = 'train/images'
data['val']   = 'valid/images'

with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data, f, allow_unicode=True, sort_keys=False)

print('✅ YAML updated.')

In [ ]:
# ── Label Consolidation: 102 classes → 5 super-classes ───────────────────────
import glob, yaml

yaml_path = '/content/dataset/sc_project/data.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    data = yaml.safe_load(f)

SUPER_CLASSES = ['Hopper_Cicada', 'Aphid', 'Borer', 'Worm_Caterpillar', 'Beetle_Weevil']

MAPPING_RULES = {
    0: ['hopper', 'cicada', 'lygus', 'leafhopper', 'planthopper'],
    1: ['aphid'],
    2: ['borer'],
    3: ['worm', 'grub', 'spodoptera', 'moth', 'cutworm', 'armyworm', 'noctua', 'caterpillar'],
    4: ['beetle', 'weevil', 'cantharis', 'chafer', 'flea']
}

old_to_new = {}
unmapped_classes = []
for i, name in enumerate(data['names']):
    name_lower = name.lower()
    mapped = False
    for new_id, keywords in MAPPING_RULES.items():
        if any(kw in name_lower for kw in keywords):
            old_to_new[i] = new_id
            mapped = True
            break
    if not mapped:
        unmapped_classes.append((i, name))

print(f'Mapped: {len(old_to_new)} classes')
print(f'Dropped (unmapped): {len(unmapped_classes)} classes')
if unmapped_classes[:5]:
    print(f'  Example dropped: {[n for _,n in unmapped_classes[:5]]}')

# Rewrite all label files
total_boxes, kept_boxes = 0, 0
label_files = glob.glob('/content/dataset/sc_project/**/labels/*.txt', recursive=True)
for txt_file in label_files:
    with open(txt_file, 'r') as f:
        lines = f.readlines()
    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if not parts: continue
        total_boxes += 1
        old_id = int(parts[0])
        if old_id in old_to_new:
            new_lines.append(f"{old_to_new[old_id]} {' '.join(parts[1:])}\n")
            kept_boxes += 1
    with open(txt_file, 'w') as f:
        f.writelines(new_lines)

print(f'Boxes — Total: {total_boxes}, Kept: {kept_boxes}, Dropped: {total_boxes - kept_boxes}')

# Write balanced YAML
balanced_yaml_path = '/content/dataset/sc_project/data_balanced.yaml'
new_data = {
    'path': '/content/dataset/sc_project',
    'train': 'train/images',
    'val': 'valid/images',
    'nc': 5,
    'names': SUPER_CLASSES
}
with open(balanced_yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(new_data, f, allow_unicode=True, sort_keys=False)

print(f'✅ Labels consolidated → 5 classes.')
print(f'   Saved: {balanced_yaml_path}')

## 🧠 PHASE 2: CNN Training (MobileNetV2 — Classification Backbone)

In [ ]:
# ── MobileNetV2 Fine-Tuning on 5 Pest Classes ────────────────────────────────
import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_CLASSES = 5
TRAIN_DIR   = '/content/dataset/sc_project/train/images'
VAL_DIR     = '/content/dataset/sc_project/valid/images'

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

# ── Data Generators ───────────────────────────────────────────────────────────
# Training augmentation to improve generalization
train_gen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)
val_gen = ImageDataGenerator(rescale=1.0/255)

# NOTE: CNN uses image-level labels derived from folder structure
# We build class folders from YOLO bounding box crops
print('\nBuilding CNN training crops from YOLO bounding box labels...')

import glob, shutil
import numpy as np
from PIL import Image

CNN_TRAIN_DIR = '/content/cnn_dataset/train'
CNN_VAL_DIR   = '/content/cnn_dataset/val'
CLASS_NAMES   = ['Hopper_Cicada', 'Aphid', 'Borer', 'Worm_Caterpillar', 'Beetle_Weevil']

for split, img_dir, lbl_dir, out_dir in [
    ('train', TRAIN_DIR, TRAIN_DIR.replace('images', 'labels'), CNN_TRAIN_DIR),
    ('val',   VAL_DIR,   VAL_DIR.replace('images', 'labels'),   CNN_VAL_DIR)
]:
    for cls in CLASS_NAMES:
        os.makedirs(f'{out_dir}/{cls}', exist_ok=True)

    img_files = glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png') + glob.glob(f'{img_dir}/*.jpeg')
    crop_count = 0
    for img_path in img_files:
        base = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = f'{lbl_dir}/{base}.txt'
        if not os.path.exists(lbl_path):
            continue
        try:
            img = Image.open(img_path).convert('RGB')
            iw, ih = img.size
            with open(lbl_path, 'r') as f:
                for j, line in enumerate(f.readlines()):
                    parts = line.strip().split()
                    if len(parts) < 5: continue
                    cls_id = int(parts[0])
                    if cls_id >= NUM_CLASSES: continue
                    cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    # Convert YOLO normalized → pixel
                    x1 = max(0, int((cx - bw/2) * iw))
                    y1 = max(0, int((cy - bh/2) * ih))
                    x2 = min(iw, int((cx + bw/2) * iw))
                    y2 = min(ih, int((cy + bh/2) * ih))
                    if x2 - x1 < 20 or y2 - y1 < 20: continue  # skip tiny crops
                    crop = img.crop((x1, y1, x2, y2)).resize((IMG_SIZE, IMG_SIZE))
                    save_path = f'{out_dir}/{CLASS_NAMES[cls_id]}/{base}_{j}.jpg'
                    crop.save(save_path, quality=85)
                    crop_count += 1
        except Exception:
            continue
    print(f'  {split}: {crop_count} pest crops saved.')

print('✅ CNN dataset built from bounding box crops.')

In [ ]:
# ── Train MobileNetV2 ─────────────────────────────────────────────────────────
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_CLASSES = 5
CNN_TRAIN_DIR = '/content/cnn_dataset/train'
CNN_VAL_DIR   = '/content/cnn_dataset/val'

train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1.0/255)

train_flow = train_datagen.flow_from_directory(
    CNN_TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
)
val_flow = val_datagen.flow_from_directory(
    CNN_VAL_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

print(f'Train samples: {train_flow.samples}')
print(f'Val samples  : {val_flow.samples}')
print(f'Class indices: {train_flow.class_indices}')

# ── Build Model ───────────────────────────────────────────────────────────────
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# PHASE A: Freeze base, train head only
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

cnn_model = Model(base_model.input, outputs)
cnn_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
cnn_model.summary(line_length=80)

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-6, monitor='val_loss')
]

print('\n[PHASE A] Training classification head (10 epochs max)...')
history_a = cnn_model.fit(
    train_flow, validation_data=val_flow,
    epochs=10, callbacks=callbacks, verbose=1
)

# PHASE B: Unfreeze top 30 layers for fine-tuning
print('\n[PHASE B] Unfreezing top 30 layers for fine-tuning...')
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

cnn_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_b = [
    EarlyStopping(patience=7, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-7, monitor='val_loss')
]

history_b = cnn_model.fit(
    train_flow, validation_data=val_flow,
    epochs=20, callbacks=callbacks_b, verbose=1
)

# Evaluate
val_loss, val_acc = cnn_model.evaluate(val_flow, verbose=0)
print(f'\n✅ CNN Final Val Accuracy: {val_acc*100:.2f}%')
print(f'   CNN Final Val Loss    : {val_loss:.4f}')

# Save
cnn_model.save('/content/cnn_pest_model.h5')
print('✅ CNN saved to /content/cnn_pest_model.h5')

## 🎯 PHASE 3: YOLOv8 Detection Training

In [ ]:
# ── YOLOv8n Training — Augmentation Boosted ───────────────────────────────────
from ultralytics import YOLO

print('Loading YOLOv8n base weights...')
yolo_model = YOLO('yolov8n.pt')

print('Starting YOLOv8 training (100 epochs)...')
print('Expected time on T4 GPU: ~40-60 minutes')

results = yolo_model.train(
    data='/content/dataset/sc_project/data_balanced.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=DEVICE,
    patience=20,
    # ── Augmentation ───────────────────────────────────────────────────────
    augment=True,
    hsv_h=0.015,         # hue shift for pest color variance
    hsv_s=0.7,          # saturation
    hsv_v=0.4,          # brightness
    flipud=0.1,         # vertical flip
    fliplr=0.5,         # horizontal flip
    mosaic=1.0,         # mosaic augmentation
    mixup=0.1,          # mix-up augmentation
    copy_paste=0.1,     # copy-paste augmentation
    # ── Optimizer ─────────────────────────────────────────────────────────
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    # ── Output ────────────────────────────────────────────────────────────
    project='/content/runs',
    name='fyolo_hybrid',
    exist_ok=True,
    save=True,
    plots=True
)

print('\n✅ YOLO training complete!')
print(f'   Best weights: /content/runs/fyolo_hybrid/weights/best.pt')

In [ ]:
# ── YOLO Validation Report ────────────────────────────────────────────────────
from ultralytics import YOLO

best_pt = '/content/runs/fyolo_hybrid/weights/best.pt'
yolo_model = YOLO(best_pt)

metrics = yolo_model.val(
    data='/content/dataset/sc_project/data_balanced.yaml',
    imgsz=640,
    conf=0.25,
    verbose=True
)

print('\n' + '='*50)
print('📊 YOLO VALIDATION RESULTS')
print('='*50)
print(f'  mAP@0.5      : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)')
print(f'  mAP@0.5:0.95 : {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)')
print(f'  Precision    : {metrics.box.mp:.4f}')
print(f'  Recall       : {metrics.box.mr:.4f}')
print('='*50)

## 🔬 PHASE 4: Fuzzy Logic Fusion Engine

In [ ]:
# ── Fuzzy Logic Severity Engine (skfuzzy) ────────────────────────────────────
# NOTE: Fuzzy is applied to decision fusion, NOT image preprocessing
# It combines YOLO confidence + CNN probability + bounding box area
# to produce a calibrated severity score.

import numpy as np
import skfuzzy as fuzz
import skfuzzy.control as ctrl

# ── Universe of Discourse ─────────────────────────────────────────────────────
yolo_conf_var  = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'yolo_conf')
cnn_prob_var   = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'cnn_prob')
box_area_var   = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'box_area')   # normalized 0–1
severity_var   = ctrl.Consequent(np.arange(0, 101,  1),    'severity')

# ── Membership Functions ──────────────────────────────────────────────────────
# Input: YOLO confidence
yolo_conf_var['low']    = fuzz.trimf(yolo_conf_var.universe, [0.00, 0.00, 0.40])
yolo_conf_var['medium'] = fuzz.trimf(yolo_conf_var.universe, [0.25, 0.50, 0.75])
yolo_conf_var['high']   = fuzz.trimf(yolo_conf_var.universe, [0.60, 1.00, 1.00])

# Input: CNN classification probability
cnn_prob_var['low']     = fuzz.trimf(cnn_prob_var.universe, [0.00, 0.00, 0.40])
cnn_prob_var['medium']  = fuzz.trimf(cnn_prob_var.universe, [0.25, 0.50, 0.75])
cnn_prob_var['high']    = fuzz.trimf(cnn_prob_var.universe, [0.60, 1.00, 1.00])

# Input: bounding box area fraction (large = many/big pests)
box_area_var['small']   = fuzz.trimf(box_area_var.universe, [0.00, 0.00, 0.20])
box_area_var['medium']  = fuzz.trimf(box_area_var.universe, [0.10, 0.25, 0.50])
box_area_var['large']   = fuzz.trimf(box_area_var.universe, [0.35, 1.00, 1.00])

# Output: severity score 0–100
severity_var['low']     = fuzz.trimf(severity_var.universe, [0,   0,  40])
severity_var['medium']  = fuzz.trimf(severity_var.universe, [25,  50, 75])
severity_var['high']    = fuzz.trimf(severity_var.universe, [60, 100, 100])

# ── Fuzzy Rules ───────────────────────────────────────────────────────────────
# Strong detections from BOTH models → HIGH severity
rule1 = ctrl.Rule(yolo_conf_var['high']   & cnn_prob_var['high'],    severity_var['high'])
rule2 = ctrl.Rule(yolo_conf_var['high']   & cnn_prob_var['medium'],  severity_var['high'])
rule3 = ctrl.Rule(yolo_conf_var['medium'] & cnn_prob_var['high'],    severity_var['high'])
# Both medium → medium severity
rule4 = ctrl.Rule(yolo_conf_var['medium'] & cnn_prob_var['medium'],  severity_var['medium'])
# Either low → low severity (reduces false positives)
rule5 = ctrl.Rule(yolo_conf_var['low']    | cnn_prob_var['low'],     severity_var['low'])
# Large bounding box area = critical infestation regardless
rule6 = ctrl.Rule(box_area_var['large'],                             severity_var['high'])
# Small bbox + low conf → definitely low
rule7 = ctrl.Rule(box_area_var['small']   & yolo_conf_var['low'],    severity_var['low'])
# Medium bbox + decent detections → medium
rule8 = ctrl.Rule(box_area_var['medium']  & yolo_conf_var['medium'], severity_var['medium'])

# ── Build Control System ──────────────────────────────────────────────────────
severity_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8])
severity_sim  = ctrl.ControlSystemSimulation(severity_ctrl)

# ── Inference Function ────────────────────────────────────────────────────────
def fuzzy_severity(yolo_c: float, cnn_p: float, box_a: float) -> dict:
    """
    Compute fuzzy severity from three inputs:
    - yolo_c : YOLO detection confidence (0–1)
    - cnn_p  : CNN classification probability (0–1)
    - box_a  : Bounding box area as fraction of image area (0–1)
    Returns: {'label': str, 'score': float}
    """
    # Clip to valid range
    yolo_c = float(np.clip(yolo_c, 0.001, 0.999))
    cnn_p  = float(np.clip(cnn_p,  0.001, 0.999))
    box_a  = float(np.clip(box_a,  0.001, 0.999))

    severity_sim.input['yolo_conf'] = yolo_c
    severity_sim.input['cnn_prob']  = cnn_p
    severity_sim.input['box_area']  = box_a
    try:
        severity_sim.compute()
        score = severity_sim.output['severity']
    except Exception:
        # Fallback: weighted average
        score = (yolo_c * 40 + cnn_p * 40 + box_a * 20)

    if score < 35:
        label = '🟢 Low'
    elif score < 65:
        label = '🟡 Medium'
    else:
        label = '🔴 HIGH'

    return {'label': label, 'score': round(score, 1)}

# ── Smoke Test ────────────────────────────────────────────────────────────────
print('Fuzzy Engine — Smoke Tests')
print('─' * 40)
tests = [
    (0.9, 0.85, 0.3,  'Expect HIGH (both models confident)'),
    (0.5, 0.5,  0.15, 'Expect Medium'),
    (0.2, 0.3,  0.05, 'Expect Low (weak detections)'),
    (0.7, 0.6,  0.7,  'Expect HIGH (large infestation area)'),
]
for yc, cp, ba, desc in tests:
    result = fuzzy_severity(yc, cp, ba)
    print(f'  yolo={yc:.1f} cnn={cp:.1f} area={ba:.1f} → {result["label"]} ({result["score"]}) | {desc}')

print('\n✅ Fuzzy Logic Engine ready.')

## 🔗 PHASE 5: Hybrid Inference — CNN + YOLO + Fuzzy

In [ ]:
# ── Load Both Trained Models ──────────────────────────────────────────────────
import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

import numpy as np
import tensorflow as tf
from ultralytics import YOLO
from PIL import Image

# Load CNN
print('Loading CNN model...')
cnn_model_loaded = tf.keras.models.load_model('/content/cnn_pest_model.h5')
print('  ✅ CNN loaded')

# Load YOLO
print('Loading YOLO model...')
yolo_model_loaded = YOLO('/content/runs/fyolo_hybrid/weights/best.pt')
print('  ✅ YOLO loaded')

CLASS_NAMES_HYBRID = ['Hopper_Cicada', 'Aphid', 'Borer', 'Worm_Caterpillar', 'Beetle_Weevil']
CLASS_DISPLAY      = ['Hopper / Cicada', 'Aphid', 'Borer', 'Worm / Caterpillar', 'Beetle / Weevil']

def hybrid_predict(image_path: str, conf_threshold: float = 0.25) -> list:
    """
    Runs the full 3-stage hybrid pipeline:
    1. YOLO detection  → bounding boxes + per-box confidence
    2. CNN global      → image-level class probability
    3. Fuzzy fusion    → severity score per detection
    Returns list of dicts with all three model outputs.
    """
    # ── Stage 1: YOLO ─────────────────────────────────────────────────────────
    yolo_results = yolo_model_loaded(image_path, conf=conf_threshold, verbose=False)[0]

    # ── Stage 2: CNN global classification ────────────────────────────────────
    pil_img = Image.open(image_path).convert('RGB').resize((224, 224))
    img_arr = np.expand_dims(np.array(pil_img, dtype=np.float32) / 255.0, axis=0)
    cnn_probs = cnn_model_loaded.predict(img_arr, verbose=0)[0]       # shape: (5,)
    cnn_top_cls  = int(np.argmax(cnn_probs))
    cnn_top_prob = float(cnn_probs[cnn_top_cls])

    # Image dimensions for area calculation
    iw, ih = pil_img.size
    img_area = iw * ih

    # ── Stage 3: Fuzzy fusion per YOLO detection ──────────────────────────────
    final_results = []
    boxes = yolo_results.boxes

    if boxes is None or len(boxes) == 0:
        return []

    for box in boxes:
        yolo_cls   = int(box.cls[0])
        yolo_c     = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        # Box area as fraction of image area
        box_w = (x2 - x1)
        box_h = (y2 - y1)
        box_area_frac = min(1.0, (box_w * box_h) / img_area)

        # CNN probability for the YOLO-detected class
        cnn_p_for_class = float(cnn_probs[yolo_cls]) if yolo_cls < len(cnn_probs) else cnn_top_prob

        # Fuzzy severity
        fuzz_result = fuzzy_severity(yolo_c, cnn_p_for_class, box_area_frac)

        # Combined confidence = weighted average of YOLO and CNN
        combined_conf = round(0.6 * yolo_c + 0.4 * cnn_p_for_class, 4)

        final_results.append({
            'class_id'       : yolo_cls,
            'class_name'     : CLASS_DISPLAY[yolo_cls] if yolo_cls < len(CLASS_DISPLAY) else f'Class {yolo_cls}',
            'bbox'           : [round(x1,1), round(y1,1), round(x2,1), round(y2,1)],
            # Per-model scores (shown separately in UI)
            'yolo_conf'      : round(yolo_c, 4),
            'cnn_prob'       : round(cnn_p_for_class, 4),
            'combined_conf'  : combined_conf,
            # Fuzzy output
            'fuzzy_severity' : fuzz_result['label'],
            'fuzzy_score'    : fuzz_result['score'],
            'box_area_frac'  : round(box_area_frac, 4),
        })

    # Sort by combined confidence descending
    final_results.sort(key=lambda d: d['combined_conf'], reverse=True)
    return final_results

print('✅ Hybrid pipeline ready.')
print("Call: hybrid_predict('/path/to/image.jpg')")

In [ ]:
# ── Demo: Run Hybrid Inference on Validation Images ──────────────────────────
import glob, os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

COLORS = ['#10b981', '#f59e0b', '#ef4444', '#8b5cf6', '#06b6d4']

# Pick 4 random validation images
val_imgs = glob.glob('/content/dataset/sc_project/valid/images/*.jpg')
val_imgs += glob.glob('/content/dataset/sc_project/valid/images/*.png')
sample_imgs = val_imgs[:4] if len(val_imgs) >= 4 else val_imgs

if not sample_imgs:
    print('No validation images found — run on any image with hybrid_predict(path)')
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.patch.set_facecolor('#0f172a')
    axes = axes.flatten()

    for idx, img_path in enumerate(sample_imgs):
        ax = axes[idx]
        img = np.array(Image.open(img_path).convert('RGB'))
        ax.imshow(img)
        ax.set_facecolor('#0f172a')
        ax.set_xticks([]); ax.set_yticks([])
        ax.spines[:].set_color('#1e293b')

        try:
            detections = hybrid_predict(img_path)
        except Exception as e:
            ax.set_title(f'Error: {e}', color='red', fontsize=8)
            continue

        for det in detections:
            x1, y1, x2, y2 = det['bbox']
            cls_id = det['class_id']
            color  = COLORS[cls_id % len(COLORS)]
            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)
            label = f"{det['class_name']} Y:{det['yolo_conf']:.2f} C:{det['cnn_prob']:.2f}"
            ax.text(x1+2, y1-5, label, color='white', fontsize=6,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))

        n = len(detections)
        maxsev = detections[0]['fuzzy_severity'] if detections else 'None'
        ax.set_title(
            f'{n} pest(s) | Severity: {maxsev}',
            color='white', fontsize=9, pad=4
        )

    plt.suptitle(
        'Hybrid Detection: CNN + YOLOv8 + Fuzzy Severity',
        color='white', fontsize=13, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.savefig('/content/hybrid_demo.png', dpi=150, bbox_inches='tight',
                facecolor='#0f172a', edgecolor='none')
    plt.show()
    print('✅ Demo saved to /content/hybrid_demo.png')

## 💾 PHASE 6: Export Weights & Download

In [ ]:
# ── Accuracy Report ───────────────────────────────────────────────────────────
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

CNN_VAL_DIR = '/content/cnn_dataset/val'
val_datagen = ImageDataGenerator(rescale=1.0/255)
val_flow = val_datagen.flow_from_directory(
    CNN_VAL_DIR, target_size=(224, 224),
    batch_size=32, class_mode='categorical', shuffle=False
)

cnn_loss, cnn_acc = cnn_model_loaded.evaluate(val_flow, verbose=0)

print('\n' + '='*55)
print('📊 FINAL HYBRID ACCURACY REPORT')
print('='*55)
print(f'  CNN (MobileNetV2) Val Accuracy : {cnn_acc*100:.2f}%')

# YOLO metrics
yolo_metrics = yolo_model_loaded.val(
    data='/content/dataset/sc_project/data_balanced.yaml',
    imgsz=640, conf=0.25, verbose=False
)
print(f'  YOLO mAP@0.5                   : {yolo_metrics.box.map50*100:.2f}%')
print(f'  YOLO mAP@0.5:0.95              : {yolo_metrics.box.map*100:.2f}%')
print(f'  YOLO Precision                 : {yolo_metrics.box.mp*100:.2f}%')
print(f'  YOLO Recall                    : {yolo_metrics.box.mr*100:.2f}%')
hybrid_score = 0.6 * yolo_metrics.box.map50 + 0.4 * cnn_acc
print(f'  ─────────────────────────────────────────────────────')
print(f'  Hybrid Combined Score           : {hybrid_score*100:.2f}%')
print('='*55)

In [ ]:
# ── Package & Download All Model Files ───────────────────────────────────────
import shutil, os
from google.colab import files

# Copy YOLO best.pt to a clean location
SRC_YOLO = '/content/runs/fyolo_hybrid/weights/best.pt'
DST_YOLO = '/content/best_fyolo_hybrid.pt'
SRC_CNN  = '/content/cnn_pest_model.h5'
DST_CNN  = '/content/cnn_pest_model.h5'

if os.path.exists(SRC_YOLO):
    shutil.copy(SRC_YOLO, DST_YOLO)
    print(f'✅ YOLO weights ready: {DST_YOLO}')
else:
    print('❌ YOLO best.pt not found — training may not have completed')

if os.path.exists(SRC_CNN):
    print(f'✅ CNN model ready    : {SRC_CNN}')
else:
    print('❌ CNN model not found — run Phase 2 first')

print('\nStarting downloads...')
print('─' * 40)

# Download YOLO weights
if os.path.exists(DST_YOLO):
    print('Downloading YOLO weights (best_fyolo_hybrid.pt)...')
    files.download(DST_YOLO)

# Download CNN model
if os.path.exists(DST_CNN):
    print('Downloading CNN model (cnn_pest_model.h5)...')
    files.download(DST_CNN)

# Download demo visualization
if os.path.exists('/content/hybrid_demo.png'):
    print('Downloading demo visualization...')
    files.download('/content/hybrid_demo.png')

print('\n✅ All files downloaded!')
print('\nNext steps for your webapp:')
print('  1. mv best_fyolo_hybrid.pt  sc_project/webapp/models/best.pt')
print('  2. mv cnn_pest_model.h5     sc_project/webapp/models/cnn_pest_model.h5')
print('  3. pip install scikit-fuzzy tensorflow')
print('  4. bash start.sh')